# B4: RoBERTa with context (workstream 5) on a Colab GPU

Trains everything in `scripts/train_b4.py` on a T4. Run the cells in order. Training runs in the background, so the status and download cells work while it trains. If the session dies, run cells 1 and 3 again: finished runs that were committed to the branch are reused.

In [ ]:
# 1. Code, data, GPU check
!git clone -q -b feat/b4-context https://github.com/qkuku1117-lgtm/emotionshift /content/emotionshift
%cd /content/emotionshift
!pip -q install -e . --no-deps
!python scripts/fetch_data.py || true
!ls data/raw/meld data/raw/dailydialog
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Smoke test (about 2 minutes): DistilRoBERTa on 12 dialogues, writes to outputs_quick/
%cd /content/emotionshift
!python scripts/train_b4.py --quick 2>&1 | grep -E '^\[|window|done|Error|Traceback|line ' 
!ls outputs_quick/reports/b4/meld/runs outputs_quick/reports/b4/dailydialog/runs

In [ ]:
# 3. Full run in the background (log: /content/train.log)
%cd /content/emotionshift
import subprocess
subprocess.Popen('nohup python scripts/train_b4.py > /content/train.log 2>&1 &', shell=True)
print('started')

In [ ]:
# 4. Status
!date; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; pgrep -f train_b4 | wc -l
!grep -E '^\[|window|done in|Error|Traceback' /content/train.log | tail -12
!tail -c 300 /content/train.log

In [ ]:
# 5. Download the results so far (small: run states and predictions)
%cd /content/emotionshift
!rm -f /content/b4_results.zip && zip -qr /content/b4_results.zip reports/b4 /content/train.log && ls -la /content/b4_results.zip
from google.colab import files
files.download('/content/b4_results.zip')